In [22]:
import logging
import os
from pathlib import Path
from types import SimpleNamespace

from MIMIC_IV_MEDS import ETL_CFG, EVENT_CFG, HAS_PRE_MEDS, RUNNER_CFG
from MIMIC_IV_MEDS import __version__ as PKG_VERSION
from MIMIC_IV_MEDS import dataset_info
from MIMIC_IV_MEDS.commands import run_command
from MIMIC_IV_MEDS.download import download_data

if HAS_PRE_MEDS:
    from MIMIC_IV_MEDS.pre_MEDS import main as pre_MEDS_transform

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


def notebook_main():
    cfg = SimpleNamespace(
        raw_input_dir="/workspaces/OpenICU-Example/output/project/mimiciv-meds-extraction-etl/raw_input",
        pre_MEDS_dir="/workspaces/OpenICU-Example/output/project/mimiciv-meds-extraction-etl/pre_MEDS",
        MEDS_cohort_dir="/workspaces/OpenICU-Example/output/project/mimiciv-meds-extraction-etl/MEDS_cohort",
        stage_runner_fp=None,
        do_download=True,
        do_demo=True,
        do_overwrite=False,
        do_copy=False,
    )

    raw_input_dir = Path(cfg.raw_input_dir)
    pre_MEDS_dir = Path(cfg.pre_MEDS_dir)
    MEDS_cohort_dir = Path(cfg.MEDS_cohort_dir)
    stage_runner_fp = cfg.stage_runner_fp

    if cfg.do_download and False:
        if cfg.do_demo:
            logger.info("Downloading demo data.")
            download_data(raw_input_dir, dataset_info, do_demo=True)
        else:
            logger.info("Downloading data.")
            download_data(raw_input_dir, dataset_info)
    else:
        logger.info("Skipping data download.")

    if HAS_PRE_MEDS and False:
        pre_MEDS_transform(
            input_dir=raw_input_dir,
            output_dir=pre_MEDS_dir,
            do_overwrite=cfg.do_overwrite,
            do_copy=cfg.do_copy,
        )
    else:
        pre_MEDS_dir = raw_input_dir

    command_parts = [
        f"DATASET_NAME={dataset_info.dataset_name}",
        f"DATASET_VERSION={dataset_info.raw_dataset_version}:{PKG_VERSION}",
        f"EVENT_CONVERSION_CONFIG_FP={str(EVENT_CFG.resolve())}",
        f"PRE_MEDS_DIR={str(pre_MEDS_dir.resolve())}",
        f"MEDS_COHORT_DIR={str(MEDS_cohort_dir.resolve())}",
        "MEDS_transform-runner",
        f"--config-path={str(RUNNER_CFG.parent.resolve())}",
        f"--config-name={RUNNER_CFG.stem}",
        f"pipeline_config_fp={str(ETL_CFG.resolve())}",
    ]

    if int(os.getenv("N_WORKERS", 1)) <= 1:
        logger.info("Running in serial mode as N_WORKERS is not set.")
        command_parts.append("~parallelize")

    if stage_runner_fp:
        command_parts.append(f"stage_runner_fp={stage_runner_fp}")

    command_parts.append("hydra.searchpath=[pkg://MEDS_transforms.configs]")

    run_command(command_parts, cfg)


notebook_main()

INFO:__main__:Skipping data download.
INFO:__main__:Running in serial mode as N_WORKERS is not set.


AttributeError: 'types.SimpleNamespace' object has no attribute 'get'